In [3]:
import sqlite3
import pandas as pd
from pathlib import Path

In [4]:
PROJECT_ROOT = Path.cwd().parent

DB_PATH = PROJECT_ROOT / "data" / "ecommerce.db"

conn = sqlite3.connect(DB_PATH)

print("Connected.")

Connected.


In [18]:
def run_sql(query):
    return pd.read_sql(query, conn)

In [5]:
query = """
CREATE VIEW IF NOT EXISTS vw_orders AS

SELECT

o.order_id,
o.customer_id,
c.customer_unique_id,

o.order_status,

o.order_purchase_timestamp,
o.order_approved_at,

o.order_delivered_carrier_date,
o.order_delivered_customer_date,

o.order_estimated_delivery_date,

c.customer_city,
c.customer_state

FROM orders o

JOIN customers c

ON o.customer_id = c.customer_id;
"""

conn.execute(query)

In [6]:
pd.read_sql(
"""
SELECT *
FROM vw_orders
LIMIT 5
""",
conn
)

,order_id,customer_id,customer_unique_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,santo andre,SP


In [7]:
query = """
CREATE VIEW IF NOT EXISTS vw_order_items AS

SELECT

oi.order_id,

oi.order_item_id,

oi.product_id,

COALESCE(
ct.product_category_name_english,
p.product_category_name,
'unknown'
) AS category,

oi.seller_id,

oi.price,

oi.freight_value,

(p.product_weight_g),

(p.product_length_cm),

(p.product_height_cm),

(p.product_width_cm)

FROM order_items oi

LEFT JOIN products p

ON oi.product_id = p.product_id

LEFT JOIN category_translation ct

ON p.product_category_name =
ct.product_category_name;
"""

conn.execute(query)

In [8]:
pd.read_sql(
"""
SELECT *
FROM vw_order_items
LIMIT 5
""",
conn
)

,order_id,order_item_id,product_id,category,seller_id,price,freight_value,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,48436dade18ac8b2bce089ec2a041202,58.90,13.29,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,pet_shop,dd7ddc04e1b6c2c614352b383efe2d36,239.90,19.93,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,furniture_decor,5b51032eddd242adc84c38acab88f23d,199.00,17.87,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,perfumery,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,garden_tools,df560393f3a51e74553ab94004ba5c87,199.90,18.14,3750.0,35.0,40.0,30.0


In [9]:
query = """
CREATE VIEW IF NOT EXISTS vw_customer_orders AS

SELECT

v.*,

SUM(p.payment_value)
AS total_payment,

MAX(p.payment_installments)
AS installments

FROM vw_orders v

LEFT JOIN payments p

ON v.order_id = p.order_id

GROUP BY

v.order_id;
"""

conn.execute(query)

In [10]:
pd.read_sql(
"""
SELECT *
FROM vw_customer_orders
LIMIT 5
""",
conn
)

,order_id,customer_id,customer_unique_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_city,customer_state,total_payment,installments
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,campos dos goytacazes,RJ,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,santa fe do sul,SP,259.83,3
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,para de minas,MG,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00,atibaia,SP,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00,varzea paulista,SP,218.04,3


In [11]:
query = """
CREATE VIEW IF NOT EXISTS vw_product_sales AS

SELECT

v.order_purchase_timestamp,

v.customer_state,

i.category,

i.product_id,

i.seller_id,

i.price,

i.freight_value,

i.price + i.freight_value
AS gross_value

FROM vw_orders v

JOIN vw_order_items i

ON v.order_id =
i.order_id;
"""

conn.execute(query)

In [12]:
pd.read_sql(
"""
SELECT *
FROM vw_product_sales
LIMIT 5
""",
conn
)

,order_purchase_timestamp,customer_state,category,product_id,seller_id,price,freight_value,gross_value
0,2017-10-02 10:56:33,SP,housewares,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,38.71
1,2018-07-24 20:41:37,BA,perfumery,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,141.46
2,2018-08-08 08:38:49,GO,auto,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,179.12
3,2017-11-18 19:28:06,RN,pet_shop,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,72.20
4,2018-02-13 21:18:39,SP,stationery,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,28.62


In [13]:
pd.read_sql(
"""
SELECT name

FROM sqlite_master

WHERE type='view'
""",
conn
)

,name
0,vw_orders
1,vw_order_items
2,vw_customer_orders
3,vw_product_sales


In [14]:
views = [
    "vw_orders",
    "vw_order_items",
    "vw_customer_orders",
    "vw_product_sales"
]

for view in views:

    rows = pd.read_sql(
        f"""
        SELECT COUNT(*) AS rows
        FROM {view}
        """,
        conn
    )

    print(view, rows.iloc[0,0])

vw_orders 99441
vw_order_items 112650
vw_customer_orders 99441
vw_product_sales 112650


In [15]:
conn.execute("DROP VIEW IF EXISTS vw_product_sales;")

In [16]:
query = """
CREATE VIEW vw_product_sales AS

SELECT

    v.order_id,

    v.order_purchase_timestamp,

    v.customer_state,

    i.category,

    i.product_id,

    i.seller_id,

    i.price,

    i.freight_value,

    i.price + i.freight_value AS gross_value

FROM vw_orders v

JOIN vw_order_items i

ON v.order_id = i.order_id;
"""

conn.execute(query)

conn.commit()

print("vw_product_sales recreated successfully.")

vw_product_sales recreated successfully.


In [19]:
run_sql("""
SELECT *
FROM vw_product_sales
LIMIT 5;
""")

,order_id,order_purchase_timestamp,customer_state,category,product_id,seller_id,price,freight_value,gross_value
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,SP,housewares,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,BA,perfumery,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,GO,auto,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,RN,pet_shop,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,SP,stationery,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,28.62
